---
### *Practical AgentOps: From PoC to Prod with MLflow 3 — ODSC AI East 2026*
---



# 4.1 — MLflow AI Gateway

---

## 💬 Product Check-in:

>**Sam:** AI + ML Engineers all have MLflow running locally. But API keys are scattered across `.env` files, every developer has their own model config, and there's no centralized way to manage costs or switch providers.

**Objective:** Use the **MLflow AI Gateway** to centralize model access behind managed endpoints

3 Ways to call endpoints you establish in your MLflow server:

1. HTTP/Invocations API
2. Completions API (OpenAI SDK) - Limited Provider Compatibility
3. Endpoint specific SDK + Passthrough API - Full Provider Compatibility

## Set Up via UI

1. Add API Keys under AI Gateway
2. Create Endpoints or use Passthrough APIs

## Create Fallbacks (Useful for Gemini Free Tier)

### Google/Gemini API Free Tier Models + Limits

| Model | Category | RPM | TPM | RPD |
|---|---|---|---|---|
| Gemini 3.1 Flash Lite (preview) | Text-out models | 15 | 250K | 500 |
| Gemini 3 Flash (preview) | Text-out models | 5 | 250K | 20 |
| Gemini 2.5 Flash | Text-out models | 5 | 0 | — |

In [ ]:
GEMINI_FREE="gemini-3.1-free-tier"
#GEMINI_PAID="gemini-2.5-flash-lite-gw" (set up live)

In [ ]:
from dotenv import load_dotenv
import requests
import os

load_dotenv()

def test_model_endpoint_http(endpoint: str, question: str) -> str:
    response = requests.post(
        f"{os.environ["MLFLOW_TRACKING_URI"]}/gateway/{endpoint}/mlflow/invocations",
        json={"messages": [{"role": "user", "content": "Hello!"}], "temperature": 0.7},
    )
    return response.json()['choices'][0]['message']['content'] 

In [ ]:
print(test_model_endpoint_http(GEMINI_FREE, "What is MLflow?"))
#print(test_model_endpoint_http(GEMINI_PAID, "Hello"))

With the AI Gateway endpoint + usage tracking enabled, we get automatic tracing, endpoint budgets, traffic split, and fallbacks without even importing MLflow into this notebook.

## Request Format + Compatibility

The MLflow Invocations API supports both OpenAI-style chat completions and embeddings endpoints. The request body follows the OpenAI chat completions format with these supported parameters.

![](../../assets/ai_gateway_completions.png)

# Calling with OpenAI Client

In [ ]:
from openai import OpenAI

openai_client = OpenAI(
    base_url=f"{os.environ["MLFLOW_TRACKING_URI"]}/gateway/mlflow/v1",
    api_key="" # not needed, configured server-side
)

SYSTEM_PROMPT = """You are an MLflow assistant."""

# From Experiment 1
def mlflow_agent(question: str) -> str:
    response = openai_client.chat.completions.create(
        model=GEMINI_FREE,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": question},
        ],
        temperature=0,
        max_completion_tokens=512,
    )
    return response.choices[0].message.content

In [ ]:
eval_dataset = [
    {
        "inputs": {"question": "How do I log a metric in MLflow?"},
        "expectations": {
            "expected_facts": ["mlflow.log_metric", "key", "value"]
        },
    },
]

In [ ]:
import mlflow

if not os.getenv("MLFLOW_TRACKING_URI"):
    mlflow.set_tracking_uri("http://127.0.0.1:5000")
# Create experiment 4
mlflow.set_experiment(os.getenv("EXPERIMENT_4_NAME","mlflow-agent-4"))

In [ ]:
# Evaluate
from mlflow.genai import evaluate
from mlflow.genai.scorers import Correctness

validation_results = evaluate(
    data=eval_dataset,
    predict_fn=mlflow_agent,
    scorers=[Correctness(model=f"gateway:/{GEMINI_FREE}")],
)

> **What just happened:** Both the `predict_fn` calls *and* the judge scorer calls went through the gateway. The gateway handled routing to Gemini, managing the API key, and enforcing any rate limits. No API keys in code, no provider-specific logic.

# Passthrough APIs (Server-side API keys)

Use local clients through MLflow endpoint to access full API support.

## OpenAI

In [ ]:
from openai import OpenAI
# Requires OpenAI API Key/endpoint configured
client = OpenAI(
    base_url=f"{os.environ["MLFLOW_TRACKING_URI"]}/gateway/openai/v1",
    api_key="dummy",  # API key not needed, configured server-side
)

# Chat Completions API
response = client.chat.completions.create(
    model="mlflow-openai-endpoint", messages=[{"role": "user", "content": "Hello!"}]
)
print(response.choices[0].message.content)

# Responses API
response = client.responses.create(
    model="mlflow-openai-endpoint",
    input="Hello!",
)
print(response.output_text)

## Gemini

**Must use corresponding SDK to make this work**

1. `uv add google`
2. `uv sync`

In [ ]:
from google import genai

client = genai.Client(
    api_key="dummy",
    http_options={
        "base_url": f"{os.environ["MLFLOW_TRACKING_URI"]}/gateway/gemini",
    },
)

response = client.models.generate_content(
    model=GEMINI_FREE,
    contents={"text": "What is MLflow?"},
)
client.close()
print(response.candidates[0].content.parts[0].text)

### Learn More

MLflow Docs - [AI Gateway](https://mlflow.org/docs/latest/genai/governance/ai-gateway/)

MLflow Docs - [Query Endpoints](https://mlflow.org/docs/latest/genai/governance/ai-gateway/endpoints/query-endpoints/)